Connecting to drive:

In [ ]:
import os
import pandas as pd

# Define your local project path ('.' means the current folder where the notebook is saved)
PROJECT_PATH = "C:\\Users\\013ri\\OneDrive\\Documents\\schoolwork\\CyberProject\\CyberSecurityMLproject"
DATASET_PATH = os.path.join(PROJECT_PATH, "datasets/training_data.csv")

# Verify the path
print(f"Project folder set to: {os.path.abspath(PROJECT_PATH)}")
print(f"Dataset path set to: {os.path.abspath(DATASET_PATH)}")

Project folder set to: C:\Users\013ri\OneDrive\Documents\schoolwork\CyberProject\CyberSecurityMLproject
Dataset path set to: C:\Users\013ri\OneDrive\Documents\schoolwork\CyberProject\CyberSecurityMLproject\datasets\training_data.csv


Generating Dataset:

In [ ]:
from faker import Faker
import random
import time
import numpy as np


def generate_dataset():
    fake = Faker()
    data = []
    NUM_SAMPLES = 50000

    # --- VOCABULARY ---
    # High Value Signals
    high_exts = [".pem", ".key", ".kdbx", ".xlsx", ".pdf", ".docx", ".wallet", ".sql"]
    high_keywords = [
        "password",
        "budget",
        "secret",
        "salary",
        "invoice",
        "prod_key",
        "backup",
        "private",
        "ssh",
    ]
    high_paths = [
        "C:\\Users\\Admin\\Desktop",
        "C:\\Users\\CEO\\Documents",
        "D:\\Backups",
    ]

    # Junk Signals (Custom lists are still good for System files)
    junk_exts = [".dll", ".sys", ".tmp", ".cache", ".png", ".ico", ".dat", ".bin"]
    junk_paths = [
        "C:\\Windows\\System32",
        "C:\\Program Files",
        "C:\\Users\\Appdata\\Local\\Temp",
    ]
    print("Generating rows...")

    for _ in range(NUM_SAMPLES):
        # 1. Decide Class (95% Junk, 5% Value)
        is_target = random.choices([True, False], weights=[5, 95], k=1)[0]

        if is_target:
            # --- HIGH VALUE GENERATION ---
            # Use Custom Lists to ensure Strong Signal
            base_name = random.choice(high_keywords)
            # Add randomness so they aren't duplicates
            name = f"{base_name}_{random.randint(10,99)}"
            ext = random.choice(high_exts)
            path = random.choice(high_paths)

            # Valuable files are usually somewhat recent
            days_old = random.randint(0, 100)
            label = 1.0

        else:
            # --- JUNK GENERATION (Hybrid) ---
            # Flip a coin: 50% "Professional Noise" (Faker), 50% "System Junk" (Custom)

            if random.random() > 0.5:
                # Option A: Faker (Realism)
                # Generates: 'marketing_plan.pdf', 'logo.png'
                name = fake.file_name(category=None, extension="")
                ext = "." + fake.file_extension()
                # Faker paths look like: /usr/local/bin... we need to fake Windows paths
                path = f"C:\\Users\\{fake.user_name()}\\Downloads"

            else:
                # Option B: System Junk
                name = f"sys_{random.randint(1000,9999)}"
                ext = random.choice(junk_exts)
                path = random.choice(junk_paths)

            days_old = random.randint(0, 2000)
            label = 0.0

        # 2. Shared Metadata Logic
        current_time = time.time()
        mod_time = int(current_time - (days_old * 24 * 3600))

        # Calculate size based on extension
        if ext in [".pem", ".key", ".txt"]:
            size = random.randint(100, 4000)  # Small keys
        else:
            size = random.randint(10000, 50 * 1024 * 1024)  # Big binaries/docs

        data.append(
            {
                "filename": name + ext,
                "path": path,
                "extension": ext,
                "size_bytes": size,
                "mod_time_unix": mod_time,
                "label": label,
            }
        )

    return pd.DataFrame(data)

Saving/loading Dataset:

In [ ]:
# Check if dataset exists
if not os.path.exists(DATASET_PATH):
    print("⚠️ Dataset not found. Generating new data (This takes time)...")

    df = generate_dataset()

    # --- SAVE TO DRIVE ---
    print(f"Saving to {DATASET_PATH}...")
    df.to_csv(DATASET_PATH, index=False)
    print("✅ Saved!")

df.head()

⚠️ Dataset not found. Generating new data (This takes time)...
Generating rows...
Saving to C:\Users\013ri\OneDrive\Documents\schoolwork\CyberProject\CyberSecurityMLproject\datasets/training_data.csv...
✅ Saved!


,filename,path,extension,size_bytes,mod_time_unix,label
0,paper.js,C:\Users\ubutler\Downloads,.js,25770189,1621424970,0.0
1,sys_3758.tmp,C:\Program Files,.tmp,8588014,1730202570,0.0
2,sys_7886.sys,C:\Program Files,.sys,28509871,1720525770,0.0
3,sys_7833.tmp,C:\Program Files,.tmp,26387708,1749296970,0.0
4,stage.js,C:\Users\shahpaul\Downloads,.js,6673146,1706788170,0.0


Pre-processing the dataset:

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import HashingVectorizer
from scipy.sparse import hstack

df = pd.read_csv(DATASET_PATH)

# recency score - new files get close to 1, year old files are close to 0
df["recency_score"] = 1 / ((time.time() - df["mod_time_unix"]) / (24 * 3600) + 1)
df["size_logged"] = np.log1p(df["size_bytes"])

# change to more nuanced scoring/hashing
df["valuable_ext"] = (
    df["extension"]
    .isin([".pem", ".key", ".kdbx", ".xlsx", ".pdf", ".docx", ".wallet", ".sql"])
    .astype(int)
)

# Initialize HashingVectorizer
hashing_vectorizer = HashingVectorizer(
    n_features=1024,
    analyzer="char",
    norm=None,
    alternate_sign=False,
    ngram_range=(3, 3),
)

# Apply feature hashing to the 'filename' column
filename_vectors = hashing_vectorizer.fit_transform(df["filename"])
path_vectors = hashing_vectorizer.fit_transform(df["path"])

# Dropping original columns after they've been used for feature creation
df.drop(
    ["filename", "path", "extension", "size_bytes", "mod_time_unix"],
    axis=1,
    inplace=True,
)

numerical_features = df[["recency_score", "size_logged", "valuable_ext"]].values
X = hstack([numerical_features, filename_vectors, path_vectors])
y = df["label"].values

In [ ]:
# To see a sample of X, we can convert a few rows to a dense array and display them.
# For example, let's look at the first 5 rows.
# Note: Converting the entire sparse matrix to a dense array can consume a lot of memory for very large datasets.

# Convert the first 5 rows to a dense array
X_sample_dense = X.toarray()

# To make it more readable, let's put it into a DataFrame, although this is only for display purposes
# as X itself is a sparse matrix.
num_numerical_features = numerical_features.shape[1]
num_filename_features = filename_vectors.shape[1]
num_path_features = path_vectors.shape[1]

# Create column names for better readability
column_names = (
    ["recency_score", "size_logged", "valuable_ext"]
    + [f"filename_hash_{i}" for i in range(num_filename_features)]
    + [f"path_hash_{i}" for i in range(num_path_features)]
)

# Display the sample as a DataFrame
X_sample_df = pd.DataFrame(X_sample_dense, columns=column_names)
display(X_sample_df)

,recency_score,size_logged,valuable_ext,filename_hash_0,filename_hash_1,filename_hash_2,filename_hash_3,filename_hash_4,filename_hash_5,filename_hash_6,...,path_hash_1014,path_hash_1015,path_hash_1016,path_hash_1017,path_hash_1018,path_hash_1019,path_hash_1020,path_hash_1021,path_hash_1022,path_hash_1023
0,0.000597,17.064729,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.002398,15.965878,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.001890,17.165761,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.005102,17.088409,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.001453,15.713602,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,0.039995,7.336286,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49996,0.012345,16.637644,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49997,0.022221,16.952557,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49998,0.011363,16.111449,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Training:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# 1. Split Data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. Train Model
# Using 'lbfgs' solver is standard. max_iter increased to ensure convergence.
model = LogisticRegression(solver="lbfgs", max_iter=1000)
model.fit(X_train, y_train)

# 3. Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 1.0

Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00      9511
         1.0       1.00      1.00      1.00       489

    accuracy                           1.00     10000
   macro avg       1.00      1.00      1.00     10000
weighted avg       1.00      1.00      1.00     10000

